# Alaska — Title 21 (Insurance) → `data/alaska/ins_codes/*.md`

The Alaska Legislature publishes statutes through **`basis/statutes.asp`**. The main page is interactive, but the same endpoint returns **HTML fragments** when called like the site’s own scripts:

- **`?media=print&type=fetch&secStart=…&secEnd=22`** — loads chunks of **Title 21** (end sentinel is **22**, i.e. start of Title 22 / Motor Vehicles).
- Responses include **`FirstSec`** and **`LastSec`** headers; this notebook **chains** `secStart = LastSec` until a chunk reaches **Title 22**.

Each **`Sec. 21.xx.xxx`** block is split on `<a name="21.xx.xxx">` anchors, converted to plain text, and saved as **`AKS_sec_21_xx_xxx.md`**.

**Official source:** [Alaska Statutes — statutes.asp](https://www.akleg.gov/basis/statutes.asp) (deep link per section: `#21.xx.xxx`).

**Politeness:** **0.2 s** delay between chunk requests (~18 calls). Adjust if needed.

Then run **`python -m app.ingest`** from the project root.

In [1]:
%pip install -q httpx beautifulsoup4

You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path

import httpx
from bs4 import BeautifulSoup

STATUTES_URL = "https://www.akleg.gov/basis/statutes.asp"
TITLE_END = "22"  # fetch range ends before Title 22
OUT_DIR = Path("data") / "alaska" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-AK-Title21/1.0 (public Alaska Statutes; educational indexing)"
REQUEST_DELAY_SEC = 0.2
TIMEOUT = 120.0
MAX_CHUNKS = 30
# Set True only if you need a local copy of concatenated HTML for debugging (can be large).
SAVE_RAW_HTML = False

SEC_ANCHOR = re.compile(r'<a name="(21\.\d+\.\d+)">', re.I)

In [3]:
def fetch_chunk(client: httpx.Client, sec_start: str) -> tuple[str, str | None, str | None]:
    """Return (html_body, FirstSec header, LastSec header)."""
    r = client.get(
        STATUTES_URL,
        params={"media": "print", "type": "fetch", "secStart": sec_start, "secEnd": TITLE_END},
    )
    r.raise_for_status()
    time.sleep(REQUEST_DELAY_SEC)
    return r.text, r.headers.get("FirstSec"), r.headers.get("LastSec")


def download_title21_html() -> str:
    parts: list[str] = []
    start = "21"
    with httpx.Client(
        headers={"User-Agent": USER_AGENT},
        timeout=TIMEOUT,
        follow_redirects=True,
        http2=False,
    ) as client:
        for n in range(MAX_CHUNKS):
            html, first_sec, last_sec = fetch_chunk(client, start)
            print(f"chunk {n + 1}: secStart={start!r} FirstSec={first_sec!r} LastSec={last_sec!r} bytes={len(html)}")
            parts.append(html)
            if not last_sec or last_sec.startswith("22"):
                break
            start = last_sec
    return "\n".join(parts)


def section_positions(html: str) -> list[tuple[str, int]]:
    """Ordered (section_id, byte_offset) for first occurrence of each anchor."""
    out: list[tuple[str, int]] = []
    seen: set[str] = set()
    for m in SEC_ANCHOR.finditer(html):
        sid = m.group(1)
        if sid in seen:
            continue
        seen.add(sid)
        out.append((sid, m.start()))
    return out


def sec_id_to_filename(sid: str) -> str:
    safe = sid.replace(".", "_")
    return f"AKS_sec_{safe}.md"


def split_into_section_files(full_html: str) -> int:
    positions = section_positions(full_html)
    wrote = 0
    for i, (sec_id, pos) in enumerate(positions):
        end = positions[i + 1][1] if i + 1 < len(positions) else len(full_html)
        chunk = full_html[pos:end]
        # Last fetch chunk can include start of Title 22; strip anything from first Title-22 anchor.
        m22 = re.search(r'<a name="22\.', chunk, re.I)
        if m22:
            chunk = chunk[: m22.start()]
        body = BeautifulSoup(chunk, "html.parser").get_text("\n", strip=True)
        url = f"{STATUTES_URL}#{sec_id}"
        title = sec_id
        m = re.match(r"Sec\.\s*(21\.\d+\.\d+)\.\s*(.+)", body, re.S)
        if m:
            head = m.group(2).strip().split("\n")[0][:120]
            title = f"AS {m.group(1)} — {head}"
        md = (
            f"# {title}\n\n"
            f"**Alaska Statutes — Title 21 (Insurance)**\n\n"
            f"**Official source:** {url}\n\n"
            f"---\n\n"
            f"{body}\n"
        )
        dest = OUT_DIR / sec_id_to_filename(sec_id)
        dest.write_text(md, encoding="utf-8")
        wrote += 1
    (OUT_DIR / "_alaska_title21_section_ids.txt").write_text(
        "\n".join(s for s, _ in positions),
        encoding="utf-8",
    )
    return wrote


full = download_title21_html()
if SAVE_RAW_HTML:
    (OUT_DIR / "_title21_raw_concat.html").write_text(full[:2_000_000], encoding="utf-8")
    print("Saved first 2MB of raw HTML to _title21_raw_concat.html")
print(f"Concatenated HTML length: {len(full)} bytes")
n = split_into_section_files(full)
print(f"Wrote {n} section files to {OUT_DIR.resolve()}")

chunk 1: secStart='21' FirstSec='21.03.000' LastSec='21.09.200' bytes=129981
chunk 2: secStart='21.09.200' FirstSec='21.09.205' LastSec='21.18.090' bytes=198948
chunk 3: secStart='21.18.090' FirstSec='21.18.100' LastSec='21.24.070' bytes=226236
chunk 4: secStart='21.24.070' FirstSec='21.24.080' LastSec='21.27.810' bytes=198151
chunk 5: secStart='21.27.810' FirstSec='21.27.820' LastSec='21.34.220' bytes=125455
chunk 6: secStart='21.34.220' FirstSec='21.34.230' LastSec='21.36.460' bytes=109774
chunk 7: secStart='21.36.460' FirstSec='21.36.461' LastSec='21.42.125' bytes=137395
chunk 8: secStart='21.42.125' FirstSec='21.42.130' LastSec='21.45.070' bytes=139024
chunk 9: secStart='21.45.070' FirstSec='21.45.080' LastSec='21.51.240' bytes=130575
chunk 10: secStart='21.51.240' FirstSec='21.51.250' LastSec='21.55.330' bytes=127449
chunk 11: secStart='21.55.330' FirstSec='21.55.340' LastSec='21.66.060' bytes=134006
chunk 12: secStart='21.66.060' FirstSec='21.66.070' LastSec='21.69.310' bytes=880

## Optional

If you set **`SAVE_RAW_HTML = True`**, delete **`_title21_raw_concat.html`** after debugging (it can be large).

## Next step

`python -m app.ingest` from the repository root.